# PairRecon n_unique differences

This notebook explains an `n_unique` difference for one column without any run artifact. It reads the Pair from `pairs.xlsx`, the column mapping and system normalization rules from the catalog, and queries both engines directly over `FROM_DATE`..`TO_DATE` (the full range, no sampling).

`TEMPLATE = "group"` runs `GROUP BY column, COUNT(*)` on each engine and compares value counts. `TEMPLATE = "distinct"` runs `SELECT DISTINCT column` on each engine and returns the set difference only.

The Right side normalizes inside the SQL warehouse with `SparkTarget`'s column plan. The Left side groups raw values in its engine, then normalizes them locally in DuckDB with `normalized_value_sql`, as the pipeline does with `left.csv`.

In [ ]:
import os
import time
from datetime import date
from pathlib import Path

import duckdb
import pandas as pd
from databricks.sdk.service.sql import Disposition, Format, StatementState

from pair_recon.app.catalog import Catalog
from pair_recon.app.config import env_value, load_environment
from pair_recon.app.runner import StatementSql, pair_config, workspace
from pair_recon.core.metadata.constant import CATALOG_SCHEMA_ENV, catalog_tables
from pair_recon.core.metadata.workbook import read_pair_workbook
from pair_recon_left.backends.sources import source_from_config
from pair_recon_left.common.model.hashing import active_normalization_rules
from pair_recon_left.common.runtime.stats import _quote, normalized_value_sql
from pair_recon_left.common.sql.dates import bounds_where, quote_ident
from pair_recon_right.engine.backends.targets import SparkTarget

load_environment()
PROFILE = env_value("DBX_PROFILE", default="corp")
WAREHOUSE_ID = env_value("DBX_WAREHOUSE_ID", default="")
CATALOG_SCHEMA = env_value(CATALOG_SCHEMA_ENV, default="catalog.schema")
WORKBOOK = Path(env_value("DBX_PAIR_WORKBOOK", default="setup/pairs.xlsx")).expanduser()
PAIR_NAME = ""
COLUMN = ""
FROM_DATE = ""
TO_DATE = ""
TEMPLATE = "group"

In [ ]:
rows = {row["pair_name"]: row for row in read_pair_workbook(WORKBOOK)}
if PAIR_NAME not in rows:
    raise ValueError(f"PAIR_NAME must be one of: {', '.join(sorted(rows))}")
if TEMPLATE not in {"group", "distinct"}:
    raise ValueError("TEMPLATE must be group or distinct")
client = workspace(PROFILE)
sql = StatementSql(client, WAREHOUSE_ID)
tables = catalog_tables(CATALOG_SCHEMA)
catalog = Catalog(sql, tables.pair_meta, tables.run_status, tables.run_error, tables.progress_snapshot)
row = {**rows[PAIR_NAME], "column_map_rows": catalog.column_mappings(PAIR_NAME)}
pair = pair_config(row)
partition = str(pair.get("partition") or "day")
left_config = {**pair["left"], "partition": partition}
right_config = {**pair["right"], "partition": partition}
if COLUMN not in pair["col_map"]:
    raise ValueError(f"COLUMN must be one of: {', '.join(pair['col_map'])}")
sources = {str(left_config.get("source")).casefold(), str(right_config.get("source")).casefold()}
rules = [rule for rule in catalog.normalization_rules() if rule["source"] in {"all", *sources}]
plan_manifest = {
    "columns": [COLUMN],
    "col_map": {COLUMN: pair["col_map"][COLUMN]},
    "normalization_rules": rules,
    "normalization_date": date.today().isoformat(),
    "col_type_overrides": {
        key: value for key, value in (pair.get("col_type_overrides") or {}).items() if key == COLUMN
    },
    "column_types": {
        key: value for key, value in (pair.get("column_types") or {}).items() if key == COLUMN
    },
}
print(f"partition={partition} left={left_config.get('source')} right={right_config.get('source')}")
print(f"{COLUMN} -> {pair['col_map'][COLUMN]} rules={[rule['rule_name'] for rule in rules]}")

In [ ]:
# >>> one INLINE JSON_ARRAY statement, every inline chunk read; raises on failure <<< #
def inline_rows(statement):
    api = client.statement_execution
    response = api.execute_statement(
        statement,
        WAREHOUSE_ID,
        disposition=Disposition.INLINE,
        format=Format.JSON_ARRAY,
        wait_timeout="30s",
    )
    while response.status.state in (StatementState.PENDING, StatementState.RUNNING):
        time.sleep(2)
        response = api.get_statement(response.statement_id)
    if response.status.state != StatementState.SUCCEEDED:
        error = getattr(response.status, "error", None)
        raise RuntimeError(getattr(error, "message", None) or f"SQL finished with {response.status.state}")
    rows = []
    result = response.result
    while result is not None:
        rows.extend(result.data_array or [])
        result = (
            None
            if result.next_chunk_index is None
            else api.get_statement_result_chunk_n(response.statement_id, result.next_chunk_index)
        )
    return rows


# >>> run a bucketed statement, splitting into more hash buckets until each inline result fits <<< #
def warehouse_frame(statement_for, columns):
    buckets = 1
    while True:
        try:
            rows = []
            for bucket in range(buckets):
                rows.extend(inline_rows(statement_for(buckets, bucket)))
                print(f"bucket {bucket + 1}/{buckets}: {len(rows)} row(s)")
            return pd.DataFrame(rows, columns=columns)
        except RuntimeError as exc:
            if "byte limit" not in str(exc).lower() or buckets >= 4096:
                raise
            buckets *= 4
            print(f"inline limit hit; retrying with {buckets} buckets")


# >>> Spark-shaped frame over SQL warehouse rows <<< #
class WarehouseFrame:
    def __init__(self, rows):
        self.rows = [tuple(item.values()) for item in rows]

    def collect(self):
        return self.rows

    def toLocalIterator(self):
        return iter(self.rows)


# >>> spark.sql stand-in that runs through the SQL warehouse <<< #
class WarehouseSpark:
    def sql(self, statement):
        return WarehouseFrame(sql.execute(statement))


target = SparkTarget(WarehouseSpark(), right_config)
health = target.health_check()
if not health["ok"]:
    raise RuntimeError(f"Right: {health['error']}")
plan = next(item for item in target._column_plans(plan_manifest) if item["left"] == COLUMN)
right_bounds = (
    None
    if partition == "all"
    else bounds_where(
        right_config["date_col"],
        dialect="spark",
        date_type=target.config["date_type"],
        from_date=FROM_DATE or None,
        to_date=TO_DATE or None,
    )
)
right_from = f"FROM {target.relation(from_date=FROM_DATE or None, to_date=TO_DATE or None)}"
right_where = target._where(right_bounds)


# >>> Right SQL for one hash bucket of the normalized value <<< #
def right_sql(buckets, bucket):
    where = right_where if buckets == 1 else f"({right_where}) AND pmod(xxhash64({plan['text']}), {buckets}) = {bucket}"
    if TEMPLATE == "group":
        return f"SELECT {plan['text']} AS v, count(1) AS n {right_from} WHERE {where} GROUP BY 1"
    return f"SELECT DISTINCT {plan['text']} AS v {right_from} WHERE {where}"


print(right_sql(1, 0))
right = warehouse_frame(right_sql, ["v", "n"] if TEMPLATE == "group" else ["v"])
right = right.reindex(columns=["v", "n"]).fillna({"n": 1}).astype({"n": "int64"})
print(f"Right: {len(right)} normalized value(s)")

In [ ]:
source = source_from_config(left_config)
health = source.health_check()
if not health["ok"]:
    raise RuntimeError(f"Left: {health['error']}")
column = quote_ident(COLUMN, source.dialect)
left_from = (
    f"FROM {source.relation(from_date=FROM_DATE or None, to_date=TO_DATE or None)} "
    f"WHERE {source._where(source._bounds(FROM_DATE or None, TO_DATE or None))}"
)
left_sql = (
    f"SELECT {column}, COUNT(*) {left_from} GROUP BY {column}"
    if TEMPLATE == "group"
    else f"SELECT DISTINCT {column} {left_from}"
)
print(left_sql)
_names, raw_rows = source.query(left_sql, operation="n_unique_debug")
left_raw = pd.DataFrame(
    [
        (None if item[0] is None or str(item[0]) == "" else str(item[0]), int(item[1]) if len(item) > 1 else 1)
        for item in raw_rows
    ],
    columns=["raw", "n"],
)
expression = normalized_value_sql(
    _quote("raw"),
    "left",
    plan_manifest["column_types"].get(COLUMN),
    active_normalization_rules(rules, left_config.get("source") or "csv"),
    plan_manifest["normalization_date"],
    collapse=plan["numeric"],
)
left = duckdb.sql(
    f"SELECT {expression} AS v, sum(n)::BIGINT AS n FROM left_raw GROUP BY 1"
).df()
print(f"Left: {len(left_raw)} raw value(s) -> {len(left)} normalized value(s)")

In [ ]:
counts = TEMPLATE == "group"
differences = duckdb.sql(
    f"""
    SELECT coalesce(l.v, r.v) AS value, l.n AS left_n, r.n::BIGINT AS right_n,
           CASE WHEN l.n IS NULL THEN 'right_only'
                WHEN r.n IS NULL THEN 'left_only'
                WHEN l.n IS NOT DISTINCT FROM r.n::BIGINT THEN 'same'
                ELSE 'count_diff' END AS status
    FROM "left" l FULL JOIN "right" r ON l.v IS NOT DISTINCT FROM r.v
    """
).df()
differences = differences[
    differences["status"].isin(["left_only", "right_only", "count_diff"] if counts else ["left_only", "right_only"])
].sort_values(["status", "value"], na_position="first")
if not counts:
    differences = differences.drop(columns=["left_n", "right_n"])
print(f"n_unique left={left['v'].notna().sum()} right={right['v'].notna().sum()}")
display(differences.groupby("status").size())
display(differences)